###Transform Circuits Data
1. Read bronze circuits table
2. Keep only the columns required for analytics (drop url column)
3. Standardise colmn names using snake case
4. Rename columns to make then more meaningful
5. Filter out rows where circuit_id is null
6. Remove duplicate records
7. Transform values of columns circuit_id and locality to Title Case
8. Write the transformed data to silver circuits table

In [0]:
%run ../00-Common/01.environment-config

In [0]:
bronze_table = f"{catalog_name}.{bronze_schema}.circuits"
silver_table = f"{catalog_name}.{silver_schema}.circuits"


In [0]:
circuits_df = spark.read.option('versionAsOf', 0).table(bronze_table)

In [0]:
display(circuits_df)

In [0]:
#circuits_select_df = circuits_df.select(
#    "circuitId",
#    "circuitName",
#    "lat",
#    "lng",
#    "locality",
#    "country",
#    "current_timestamp",
#    "source_file"
#)

In [0]:
from pyspark.sql import functions as f

In [0]:
circuits_select_df = circuits_df.select(
    f.col("circuitId"),
    f.col("circuitName"),
    f.col("lat"),
    f.col("lng"),
    f.col("locality"),
    f.col("country"),
    f.col("current_timestamp"),
    f.col("source_file")
)

In [0]:
display(circuits_select_df)

In [0]:
#circuits_renamed_df = (
#    circuits_select_df
#    .withColumnRenamed("circuitId", "circuit_id")
#    .withColumnRenamed("circuitName", "circuit_name")
#    .withColumnRenamed("lat", "latitude")
#    .withColumnRenamed("lng", "longitude")
#)

In [0]:
circuits_renamed_df = (
    circuits_select_df
    .withColumnsRenamed({
        "circuitId": "circuit_id",
        "circuitName": "circuit_name",
        "lat": "latitude",
        "lng": "longitude"
        })
)

In [0]:
display(circuits_renamed_df)

In [0]:
#circuits_valid_df = (
#    circuits_renamed_df.filter(
#       "circuit_id IS NOT NULL"
#    )
#)

In [0]:
circuits_valid_df = (
    circuits_renamed_df.filter(
       f.col("circuit_id").isNotNull()
    )
)

In [0]:
display(circuits_valid_df)

In [0]:
#circuits_distinct_df = circuits_valid_df.distinct()

In [0]:
circuits_distinct_df = circuits_valid_df.dropDuplicates(["circuit_id"])

In [0]:
display(circuits_distinct_df)

In [0]:
circuits_final_df = (
    circuits_distinct_df
        .withColumn('circuit_name',f.initcap(f.col("circuit_name")))
        .withColumn('locality', f.initcap(f.col("locality")))

)

In [0]:
display(circuits_final_df)

In [0]:
(
    circuits_final_df
    .write
    .mode("overwrite")
    .format("delta")
    .saveAsTable(silver_table)
)

In [0]:
display(spark.table(silver_table))

In [0]:
%sql
select * from formula1.silver.circuits